In [3]:
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [6]:
documents = documents_llm

In [7]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
import json
user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
response.output_parsed.questions

['Can I still join the course if I only found it recently?',
 'Is it okay to join late, or is it too late to start now?',
 'If I start the course after the official start date, can I still get a certificate?',
 'What do I need to do to qualify for a certificate if I join now?',
 'Am I allowed to enroll even though the course has already been going on for a while?']

In [16]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [17]:
from evaluation_utils import llm_structured

In [18]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I found it late?', 'Am I allowed to start the course after it has already begun?', 'Is it too late to join llm-zoomcamp now?', 'If I join late, can I still get a certificate somehow?', 'What do I need to do to be eligible for a certificate if I discovered the course late?']


In [19]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=86, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=293)

In [20]:
from evaluation_utils import calc_price

In [21]:
calc_price(usage)

{'input_cost': 0.00015525,
 'output_cost': 0.00038700000000000003,
 'total_cost': 0.00054225}

In [22]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to start the course after it has already begun?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to join llm-zoomcamp now?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for a certificate if I discovered the course late?',
  'document': '74eb249bbf'}]

In [23]:
import pandas as pd

In [24]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,Am I allowed to start the course after it has ...,74eb249bbf
2,Is it too late to join llm-zoomcamp now?,74eb249bbf
3,"If I join late, can I still get a certificate ...",74eb249bbf
4,What do I need to do to be eligible for a cert...,74eb249bbf


In [25]:
from evaluation_utils import llm_structured_retry

In [26]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [27]:
generate_ground_truth(doc)

([{'question': 'Can I still join the course if I just found out about it?',
   'document': '74eb249bbf'},
  {'question': 'Is it okay to join late, or do I need to start from the beginning?',
   'document': '74eb249bbf'},
  {'question': 'If I join now, will I still be able to get a certificate?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to qualify for the course certificate?',
   'document': '74eb249bbf'},
  {'question': "If I'm joining after the course started, is the project submission deadline the only thing I need to worry about?",
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=304))

In [28]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [29]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [30]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [33]:
ground_truth[10]

{'question': 'Where can I watch the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [34]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07720275

In [36]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07720275

In [37]:
df_ground_truth = pd.DataFrame(ground_truth)

In [40]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [41]:
usage.input_tokens

279